In [1]:
# Cell 1 — Imports / setup
import os
import random
from pathlib import Path
from typing import Dict, List, Tuple

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

from tqdm.auto import tqdm
import pandas as pd

SEED = 123
random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
device


'cuda'

In [2]:
# Cell 2 — Config: caminhos e parâmetros
DATASET_A = "DATASET54000"
DATASET_B = "DDPM-F64x4"

BATCH_SIZE = 32
NUM_WORKERS = 1  # ajuste conforme seu PC
PIN_MEMORY = (device == "cuda")

# Normalização padrão ImageNet (para ResNet pré-treinada)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

# Transform: 
transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


In [3]:
# Cell 3 — Feature extractor (ResNet18) que aceita 64×64
@torch.no_grad()
def build_feature_extractor(device: str = "cpu") -> nn.Module:
    # ResNet18 pré-treinada; remove a FC e fica com vetor 512-d
    weights = models.ResNet18_Weights.DEFAULT
    model = models.resnet18(weights=weights)
    model.fc = nn.Identity()
    model.eval().to(device)
    return model

feature_extractor = build_feature_extractor(device)
feature_extractor


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [4]:
# Cell 4 — Helpers: carregar datasets, mapear classes e pegar índices por classe
def make_imagefolder(root: str, transform) -> datasets.ImageFolder:
    root = str(root)
    if not os.path.isdir(root):
        raise FileNotFoundError(f"Pasta não encontrada: {root}")
    return datasets.ImageFolder(root=root, transform=transform)

dsA = make_imagefolder(DATASET_A, transform)
dsB = make_imagefolder(DATASET_B, transform)

print("Classes A:", dsA.classes)
print("Classes B:", dsB.classes)

if dsA.classes != dsB.classes:
    raise ValueError(
        "As classes (nomes das subpastas) precisam ser idênticas e na mesma ordem.\n"
        f"A: {dsA.classes}\nB: {dsB.classes}"
    )

classes = dsA.classes

def indices_by_class(imagefolder: datasets.ImageFolder) -> Dict[str, List[int]]:
    out = {c: [] for c in imagefolder.classes}
    for idx, (_, y) in enumerate(imagefolder.samples):
        c = imagefolder.classes[y]
        out[c].append(idx)
    return out

idxA = indices_by_class(dsA)
idxB = indices_by_class(dsB)

{c: (len(idxA[c]), len(idxB[c])) for c in classes}


Classes A: ['BULKCARRIER', 'CONTAINERSHIP', 'GENERALCARGO', 'OILPRODUCTSTANKER', 'PASSENGERSSHIP', 'TANKER', 'TRAWLER', 'TUG', 'VEHICLESCARRIER', 'YACHT']
Classes B: ['BULKCARRIER', 'CONTAINERSHIP', 'GENERALCARGO', 'OILPRODUCTSTANKER', 'PASSENGERSSHIP', 'TANKER', 'TRAWLER', 'TUG', 'VEHICLESCARRIER', 'YACHT']


{'BULKCARRIER': (5400, 5400),
 'CONTAINERSHIP': (5400, 5400),
 'GENERALCARGO': (5400, 5400),
 'OILPRODUCTSTANKER': (5400, 5400),
 'PASSENGERSSHIP': (5400, 5400),
 'TANKER': (5400, 5400),
 'TRAWLER': (5400, 5400),
 'TUG': (5400, 5400),
 'VEHICLESCARRIER': (5400, 5400),
 'YACHT': (5400, 5400)}

In [5]:
# Cell 5 — Extrair features de uma classe (igualando tamanhos se necessário)
@torch.no_grad()
def extract_features_for_indices(
    dataset: datasets.ImageFolder,
    indices: List[int],
    model: nn.Module,
    device: str,
    batch_size: int = 128,
    num_workers: int = 2,
    pin_memory: bool = False,
) -> torch.Tensor:
    loader = DataLoader(
        Subset(dataset, indices),
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        drop_last=False,
    )

    feats = []
    for x, _ in loader:
        x = x.to(device, non_blocking=True)
        f = model(x)  # [B, 512]
        feats.append(f.detach().cpu())
    return torch.cat(feats, dim=0)

def match_class_sizes(a_indices: List[int], b_indices: List[int], seed: int = 123):
    n = min(len(a_indices), len(b_indices))
    rng = random.Random(seed)
    a_sel = rng.sample(a_indices, n) if len(a_indices) != n else list(a_indices)
    b_sel = rng.sample(b_indices, n) if len(b_indices) != n else list(b_indices)
    return a_sel, b_sel


In [6]:
# Cell 6 — KID = MMD² não-viesado com kernel polinomial grau 3
def polynomial_kernel(x: torch.Tensor, y: torch.Tensor, degree: int = 3) -> torch.Tensor:
    """
    x: [n, d], y: [m, d]
    k(a,b) = ((a^T b)/d + 1)^degree
    """
    d = x.shape[1]
    return ((x @ y.T) / d + 1.0) ** degree

def kid_unbiased(x: torch.Tensor, y: torch.Tensor, degree: int = 3) -> float:
    """
    Estimador não-viesado do MMD^2 (KID).
    Requer n>=2 e m>=2.
    """
    n, d = x.shape
    m, _ = y.shape
    if n < 2 or m < 2:
        return float("nan")

    Kxx = polynomial_kernel(x, x, degree=degree)
    Kyy = polynomial_kernel(y, y, degree=degree)
    Kxy = polynomial_kernel(x, y, degree=degree)

    # remove diagonal para termos i!=j
    sum_Kxx = (Kxx.sum() - torch.diagonal(Kxx).sum())
    sum_Kyy = (Kyy.sum() - torch.diagonal(Kyy).sum())

    term_xx = sum_Kxx / (n * (n - 1))
    term_yy = sum_Kyy / (m * (m - 1))
    term_xy = (2.0 * Kxy.sum()) / (n * m)

    mmd2 = term_xx + term_yy - term_xy
    return float(mmd2.item())


In [7]:
# Cell 7 — Rodar por classe e montar tabela
results = []
for c in tqdm(classes, desc="Calculando KID por classe"):
    a_idx, b_idx = match_class_sizes(idxA[c], idxB[c], seed=SEED)

    featA = extract_features_for_indices(
        dsA, a_idx, feature_extractor, device,
        batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
    )
    featB = extract_features_for_indices(
        dsB, b_idx, feature_extractor, device,
        batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
    )

    kid = kid_unbiased(featA, featB, degree=3)

    results.append({
        "class": c,
        "n_used": len(a_idx),
        "kid": kid,
    })

df = pd.DataFrame(results).sort_values("class").reset_index(drop=True)
df


Calculando KID por classe:   0%|          | 0/10 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/conv.py:459: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:80.)
  return F.conv2d(input, weight, bias, self.stride,


,class,n_used,kid
0,BULKCARRIER,5400,1.585060
1,CONTAINERSHIP,5400,1.479085
2,GENERALCARGO,5400,1.005194
3,OILPRODUCTSTANKER,5400,1.337236
4,PASSENGERSSHIP,5400,1.927391
5,TANKER,5400,1.390396
6,TRAWLER,5400,1.131665
7,TUG,5400,0.745111
8,VEHICLESCARRIER,5400,2.094446
9,YACHT,5400,0.627058


In [8]:
# Cell 8 — Resumo (média do KID entre classes)
df["kid"].mean(), df["kid"].std()


(1.332264232635498, 0.4729627365100775)